# Libraries

In [62]:
import json
import time
import numpy as np
from pathlib import Path
from typing import List, Dict, Any
import re
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder
try:
    import faiss
    print("✓ FAISS loaded")
except ImportError:
    raise ImportError("Run: pip install faiss-cpu")

print("✓ Loading Reranker ...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

✓ FAISS loaded
✓ Loading Reranker ...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4758.27it/s]


# Paths

In [63]:
BASE_DIR    = Path("../")
CHUNK_DIR   = BASE_DIR / "data" / "chunks"
EMB_DIR     = BASE_DIR / "data" / "gold"
INDEX_DIR   = BASE_DIR / "data" / "indexes"
META_DIR    = BASE_DIR / "data" / "metadata"

EMB_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_FILES = {
    "small_fixed": CHUNK_DIR / "strategy_A_small_fixed.json",
    "large_fixed": CHUNK_DIR / "strategy_B_large_fixed.json",
    "semantic":    CHUNK_DIR / "strategy_C_semantic.json",
}

MODELS = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "bge":    "BAAI/bge-small-en-v1.5",
}

print("✓ Paths configured")
for name, path in CHUNK_FILES.items():
    exists = "✓" if path.exists() else "✗ MISSING"
    print(f"  {exists} {name}: {path.name}")

✓ Paths configured
  ✓ small_fixed: strategy_A_small_fixed.json
  ✓ large_fixed: strategy_B_large_fixed.json
  ✓ semantic: strategy_C_semantic.json


# Load Chunks

In [64]:
chunks_by_strategy: Dict[str, List[Dict]] = {}

for strategy_name, path in CHUNK_FILES.items():
    with open(path, encoding="utf-8") as f:
        chunks = json.load(f)
    chunks_by_strategy[strategy_name] = chunks
    print(f"  ✓ {strategy_name}: {len(chunks)} chunks loaded")

print(f"\nTotal chunks across all strategies: "
      f"{sum(len(v) for v in chunks_by_strategy.values())}")

  ✓ small_fixed: 1515 chunks loaded
  ✓ large_fixed: 595 chunks loaded
  ✓ semantic: 791 chunks loaded

Total chunks across all strategies: 2901


# Load Embedding Models

In [65]:
models: Dict[str, SentenceTransformer] = {}

for model_key, model_name in MODELS.items():
    print(f"  Loading {model_name} ...")
    t0 = time.time()
    models[model_key] = SentenceTransformer(model_name)
    print(f"  ✓ {model_key} loaded in {time.time() - t0:.1f}s")

print("\n✓ Both models ready")

  Loading sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5504.83it/s]


  ✓ minilm loaded in 2.3s
  Loading BAAI/bge-small-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5374.44it/s]


  ✓ bge loaded in 2.3s

✓ Both models ready


# Generate Embeddings — 6 Combinations

In [66]:
def generate_and_save_embeddings(
    model: SentenceTransformer,
    chunks: List[Dict],
    save_path: Path,
    batch_size: int = 64,
) -> np.ndarray:
    """
    Encode chunk texts, L2-normalize, and save to disk.
    Returns the normalized embedding matrix (n_chunks, dim).
    """
    texts = [c["text"] for c in chunks]

    t0 = time.time()
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,   # L2-normalize here
    )
    elapsed = time.time() - t0

    np.save(save_path, embeddings)
    print(f"    Shape: {embeddings.shape} | Time: {elapsed:.1f}s | "
          f"Speed: {len(texts)/elapsed:.0f} chunks/s → saved to {save_path.name}")

    return embeddings


# --- Generate all 6 combinations ---
embeddings_store: Dict[str, np.ndarray] = {}  # key: "minilm_small_fixed", etc.

for model_key, model in models.items():
    print(f"\n{'='*55}")
    print(f"Model: {model_key}")
    print(f"{'='*55}")
    for strategy_name, chunks in chunks_by_strategy.items():
        key       = f"{model_key}_{strategy_name}"
        save_path = EMB_DIR / f"{key}.npy"

        if save_path.exists():
            # Load cached embeddings to avoid recomputing
            embeddings = np.load(save_path)
            print(f"  ✓ {key}: loaded from cache {embeddings.shape}")
        else:
            print(f"  Encoding {key} ({len(chunks)} chunks)...")
            embeddings = generate_and_save_embeddings(model, chunks, save_path)

        embeddings_store[key] = embeddings

print(f"\n✓ {len(embeddings_store)} embedding matrices ready")


Model: minilm
  ✓ minilm_small_fixed: loaded from cache (1515, 384)
  ✓ minilm_large_fixed: loaded from cache (595, 384)
  ✓ minilm_semantic: loaded from cache (791, 384)

Model: bge
  ✓ bge_small_fixed: loaded from cache (1515, 384)
  ✓ bge_large_fixed: loaded from cache (595, 384)
  ✓ bge_semantic: loaded from cache (791, 384)

✓ 6 embedding matrices ready


# Build FAISS Indexes

In [67]:
faiss_indexes: Dict[str, faiss.IndexFlatIP] = {}

for key, embeddings in embeddings_store.items():
    dim        = embeddings.shape[1]
    index      = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))
    faiss_indexes[key] = index

    save_path = INDEX_DIR / f"{key}.faiss"
    faiss.write_index(index, str(save_path))

    print(f"  ✓ {key}: {index.ntotal} vectors, dim={dim} → {save_path.name}")

print(f"\n✓ {len(faiss_indexes)} FAISS indexes built and saved")

  ✓ minilm_small_fixed: 1515 vectors, dim=384 → minilm_small_fixed.faiss
  ✓ minilm_large_fixed: 595 vectors, dim=384 → minilm_large_fixed.faiss
  ✓ minilm_semantic: 791 vectors, dim=384 → minilm_semantic.faiss
  ✓ bge_small_fixed: 1515 vectors, dim=384 → bge_small_fixed.faiss
  ✓ bge_large_fixed: 595 vectors, dim=384 → bge_large_fixed.faiss
  ✓ bge_semantic: 791 vectors, dim=384 → bge_semantic.faiss

✓ 6 FAISS indexes built and saved


# Retrieval Pipeline

In [68]:
def retrieve(
    query: str,
    model_key: str,
    strategy_name: str,
    k: int = 5,
) -> List[Dict[str, Any]]:
    """
    Pipeline de dos etapas: 
    1. FAISS (Bi-Encoder) para recuperar candidatos (Top-50).
    2. Reranker (Cross-Encoder) para reordenar esos candidatos.
    """
    index_key = f"{model_key}_{strategy_name}"
    model     = models[model_key]
    index     = faiss_indexes[index_key]
    chunks    = chunks_by_strategy[strategy_name]

    # Retrieve candidates with FAISS
    q_vec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, indices = index.search(q_vec, 20) # Recover 50 candidates for reranking

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        result = dict(chunks[idx])
        result["score"] = float(score)
        results.append(result)

    # Reranking (Cross-Encoder)
    if results:
        # Prepare pairs for Reranker
        pairs = [[query, res['text']] for res in results]
        # Obtain scores from Cross-Encoder
        rerank_scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
        
        # Asign new scores from Reranker
        for i, r_score in enumerate(rerank_scores):
            results[i]["score"] = float(r_score)
        
        # Re-order based on Reranker score
        results = sorted(results, key=lambda x: x["score"], reverse=True)

    # Return the final Top K
    return results[:k]

# Evaluation — Recall@K, Precision@K, MRR and Evaluation Proposal

In [69]:
BENCHMARK_PATH = Path("../data/benchmark/benchmark.json")
BENCHMARK = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))

# Validate benchmark schema
REQUIRED_KEYS = {"query", "relevant_paper_id", "answer_contains", "difficulty"}
for i, item in enumerate(BENCHMARK):
    missing = REQUIRED_KEYS - set(item.keys())
    assert not missing, f"Item {i} is missing fields: {missing}"

print(f"✓ Benchmark loaded: {len(BENCHMARK)} queries")

# Adaptive Relevance Scoring (ARS)
def get_ars_score(chunk: dict, item: dict) -> float:
    """Calculates an adaptive score based on lexical, technical, and semantic signals."""
    if chunk["paper_id"] != item["relevant_paper_id"]:
        return 0.0
    
    chunk_text = chunk["text"].lower()
    
    # Lexical: Keyword overlap (ignoring stop words)
    stop_words = {"we", "the", "a", "is", "as", "with", "of", "to", "in", "by", "for", "that"}
    keywords = [w for w in re.findall(r'\w+', item["answer_contains"].lower()) if w not in stop_words]
    matches = sum(1 for kw in keywords if kw in chunk_text)
    lexical_score = (matches / len(keywords)) if keywords else 0.0
    
    # Technical: Heuristic for math/technical notation
    tech_patterns = ["y =", "f(x)", "equation", "formula", "wi", "x", "residual", "loss"]
    tech_matches = sum(1 for p in tech_patterns if p in chunk_text)
    technical_score = min(tech_matches / 4, 1.0)
    
    # Semantic: Vector similarity
    semantic_score = chunk.get("score", 0.5)
    
    # Weighted final score
    return (0.35 * lexical_score) + (0.25 * technical_score) + (0.4 * semantic_score)

def is_relevant(chunk: dict, item: dict, threshold: float = 0.45) -> bool:
    return get_ars_score(chunk, item) >= threshold

# Metrics Calculation 
def compute_metrics(
    query: str, benchmark_item: dict, model_key: str, strategy: str, k_values: List[int] = [3, 5]
) -> Dict[str, float]:
    
    results = retrieve(query, model_key, strategy, k=max(k_values))
    metrics = {}

    # MRR: 1 / rank of first relevant result
    metrics["mrr"] = 0.0
    for rank, chunk in enumerate(results, start=1):
        if is_relevant(chunk, benchmark_item):
            metrics["mrr"] = 1.0 / rank
            break

    # Recall & Precision
    for k in k_values:
        top_k = results[:k]
        hits = sum(1 for chunk in top_k if is_relevant(chunk, benchmark_item))
        metrics[f"recall@{k}"] = 1 if hits > 0 else 0
        metrics[f"precision@{k}"] = hits / k

    return metrics

✓ Benchmark loaded: 30 queries


In [70]:
eval_results = []
detail_results = []

for model in models:
    for strategy in chunks_by_strategy:
        config_name = f"{model}_{strategy}"
        config_metrics = []
        diff_bins = {"easy": [], "medium": [], "hard": []}

        for item in BENCHMARK:
            m = compute_metrics(item["query"], item, model, strategy)
            m.update({
                "query": item["query"], 
                "difficulty": item["difficulty"], 
                "config": config_name
            })
            
            config_metrics.append(m)
            diff_bins[item["difficulty"]].append(m)
            detail_results.append(m)

        # Aggregate Results
        summary = {"config": config_name, "model": model, "strategy": strategy}
        for m_key in ["mrr", "recall@3", "recall@5", "precision@5"]:
            summary[m_key] = round(np.mean([q[m_key] for q in config_metrics]), 4)
        
        # Breakdown by difficulty
        for diff in ["easy", "medium", "hard"]:
            items = diff_bins[diff]
            summary[f"mrr_{diff}"] = round(np.mean([q["mrr"] for q in items]), 4) if items else 0

        eval_results.append(summary)

# Formatted table including Easy, Medium, and Hard MRR
print(f"{'Config':<30} | {'R@3':>6} | {'R@5':>6} | {'P@5':>6} | {'MRR':>6} | {'Easy':>6} | {'Med':>6} | {'Hard':>6}")
print("-" * 105)

for r in eval_results:
    print(f"{r['config']:<30} | {r['recall@3']:>6.3f} | {r['recall@5']:>6.3f} | "
          f"{r['precision@5']:>6.3f} | {r['mrr']:>6.3f} | {r['mrr_easy']:>6.3f} | "
          f"{r['mrr_medium']:>6.3f} | {r['mrr_hard']:>6.3f}")

# Save results to disk
with open(META_DIR / "retrieval_eval.json", "w") as f:
    json.dump(eval_results, f, indent=2)

Config                         |    R@3 |    R@5 |    P@5 |    MRR |   Easy |    Med |   Hard
---------------------------------------------------------------------------------------------------------
minilm_small_fixed             |  0.767 |  0.767 |  0.580 |  0.767 |  1.000 |  0.833 |  0.375
minilm_large_fixed             |  0.767 |  0.767 |  0.600 |  0.767 |  1.000 |  0.833 |  0.375
minilm_semantic                |  0.800 |  0.800 |  0.567 |  0.800 |  1.000 |  0.833 |  0.500
bge_small_fixed                |  0.733 |  0.733 |  0.567 |  0.733 |  1.000 |  0.750 |  0.375
bge_large_fixed                |  0.733 |  0.733 |  0.553 |  0.733 |  1.000 |  0.750 |  0.375
bge_semantic                   |  0.767 |  0.767 |  0.527 |  0.767 |  1.000 |  0.750 |  0.500
